In [0]:
%sql
create external location if not exists Ext1 
url "abfss://bronze@piyushstorag1.dfs.core.windows.net"
with(storage credential cread1);

create external location if not exists Ext2
url "abfss://silver@piyushstorag1.dfs.core.windows.net"
with(storage credential cread1);

create external location if not exists Ext3 
url "abfss://gold@piyushstorag1.dfs.core.windows.net"
with(storage credential cread1);


In [0]:
%sql
drop table if exists fisrt_cat.bronze.products;
drop table if exists fisrt_cat.silver.fulldata;


##Silver Layer

In [0]:
from pyspark.sql.functions import explode
try:
    df=spark.read.json("abfss://bronze@piyushstorag1.dfs.core.windows.net/increamental/")
    df_final = df.select(explode("products").alias("product")).select("product.*")
    df_final.display()
except Exception as e:
    print(f"Error: {e}")

Error: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for JSON. It must be specified manually. SQLSTATE: 42KD9


In [0]:
%sql
create schema if not exists fisrt_cat.gold
managed location 'abfss://gold@piyushstorag1.dfs.core.windows.net';
create schema if not exists fisrt_cat.silver
managed location 'abfss://silver@piyushstorag1.dfs.core.windows.net';
create schema if not exists fisrt_cat.bronze
managed location 'abfss://bronze@piyushstorag1.dfs.core.windows.net';

In [0]:
try:
    from delta.tables import DeltaTable
    if spark.catalog.tableExists("fisrt_cat.bronze.products"):
        table="fisrt_cat.bronze.products"
        delta_table=DeltaTable.forName(spark,table)
        delta_table.alias("target").merge(
            df_final.alias("source"),
            "target.id=source.id"
        ).whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    else:
        df_final.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("fisrt_cat.bronze.products")
except Exception as e:
    print(f"Error: {e}")

In [0]:
%sql
select * from fisrt_cat.bronze.products

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",Ships in 1 week,BEA-CHI-LIP-004,91,"L

In [0]:
df=spark.read.table("fisrt_cat.bronze.products")

In [0]:
df.display()

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",Ships in 1 week,BEA-CHI-LIP-004,91,"L

In [0]:
df=df.fillna({"brand":"unknown"})
df.display()

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",Ships in 1 week,BEA-CHI-LIP-004,91,"L

In [0]:
df=df.dropDuplicates(["id"])

In [0]:
df.display()

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight,Status
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4,OVERSTOCK
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9,IN STOCK
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8,OVERSTOCK
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",S

In [0]:
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("fisrt_cat.silver.fulldata")

In [0]:
%sql
alter table fisrt_cat.silver.fulldata
add column Status string;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6606757141223173>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'alter table fisrt_cat.silver.fulldata\nadd column Status string;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:183, in SqlMagic.sql(self, line, cell)
    177 except BaseException as e:
    178     s

In [0]:
%sql
update fisrt_cat.silver.fulldata
 set
     Status = case 
        when Category = 'fragrances' and Stock < 10 then 'LOW STOCK'
        when Category = 'fragrances' and Stock > 50 then 'OVERSTOCK'
        when Category = 'furniture' and Stock > 20 then 'OVERSTOCK'
        when Stock < 5 then 'LOW STOCK'
        when Stock > 50 then 'OVERSTOCK'
        else 'IN STOCK'
    end

In [0]:
df=spark.read.table("fisrt_cat.silver.fulldata")
df.display()

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight,Status
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4,OVERSTOCK
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9,IN STOCK
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8,OVERSTOCK
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",S

In [0]:
try:
    df_produacts=df.selectExpr("id as Product_Id","title as Product_Name","brand as Brand","category as Category","price as Price","discountPercentage as Discount","rating as Rating")

    from delta.tables import DeltaTable
    table = "fisrt_cat.silver.products"
    if spark.catalog.tableExists(table):
        delta_table = DeltaTable.forName(spark, table)
        delta_table.alias("target").merge(
            df_produacts.alias("source"),
            "target.Product_Id = source.Product_Id"
        ).whenMatchedUpdate(set={
        "Product_Name": "source.Product_Name",
        "Brand": "source.Brand",
        "Category": "source.Category",
        "Price": "source.Price",
        "Discount": "source.Discount",
        "Rating": "source.Rating"
    }).whenNotMatchedInsert(values={
        "Product_Id": "source.Product_Id",
        "Product_Name": "source.Product_Name",
        "Brand": "source.Brand",
        "Category": "source.Category",
        "Price": "source.Price",
        "Discount": "source.Discount",
        "Rating": "source.Rating"
    }).execute()
    else:
        df_produacts.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table)
except Exception as e:
    print(f"Error: {e}")
  

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5355360288854455>, line 1
----> 1 df_produacts=df.selectExpr("id as Product_Id","title as Product_Name","brand as Brand","category as Category","price as Price","discountPercentage as Discount","rating as Rating")
      3 from delta.tables import DeltaTable
      4 table = "fisrt_cat.silver.products"

NameError: name 'df' is not defined

In [0]:
%sql 
select * from fisrt_cat.silver.products

Product_Id,Product_Name,Brand,Category,Price,Discount,Rating
1,Essence Mascara Lash Princess,Essence,beauty,9.99,10.48,2.56
2,Eyeshadow Palette with Mirror,Glamour Beauty,beauty,19.99,18.19,2.86
3,Powder Canister,Velvet Touch,beauty,14.99,9.84,4.64
4,Red Lipstick,Chic Cosmetics,beauty,12.99,12.16,4.36
5,Red Nail Polish,Nail Couture,beauty,8.99,11.44,4.32
6,Calvin Klein CK One,Calvin Klein,fragrances,49.99,1.89,4.37
7,Chanel Coco Noir Eau De,Chanel,fragrances,129.99,16.51,4.26
8,Dior J'adore,Dior,fragrances,89.99,14.72,3.8
9,Dolce Shine Eau de,Dolce & Gabbana,fragrances,69.99,0.62,3.96
10,Gucci Bloom Eau de,Gucci,fragrances,79.99,14.39,2.74


In [0]:
try:
    df_inventory=df.selectExpr("id as Product_id","stock as Stock","minimumOrderQuantity as Minimum_Order_Quantity","Status")
    from delta.tables import DeltaTable
    table = "fisrt_cat.silver.inventory"
    if spark.catalog.tableExists("fisrt_cat.silver.inventory"):
        
        delta_table = DeltaTable.forName(spark, table)

        delta_table.alias("target").merge(
            df_inventory.alias("source"),
            "target.Product_Id = source.Product_Id"
        ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

    else:
        df_inventory.write.format("delta") \
            .mode("overwrite") \
            .option("mergeSchema", "true")\
            .saveAsTable(table)
except Exception as e:
    print(f"Error: {e}")

    



In [0]:
df_inventory.display()

Product_id,Stock,Minimum_Order_Quantity,Status
1,99,48,OVERSTOCK
2,34,20,IN STOCK
3,89,22,OVERSTOCK
4,91,40,OVERSTOCK
5,79,22,OVERSTOCK
6,29,9,IN STOCK
7,58,1,OVERSTOCK
8,98,10,OVERSTOCK
9,4,2,LOW STOCK
10,91,2,OVERSTOCK


In [0]:
%sql
select * from fisrt_cat.silver.inventory

Product_id,Stock,Minimum_Order_Quantity,Status
1,99,48,OVERSTOCK
2,34,20,IN STOCK
3,89,22,OVERSTOCK
4,91,40,OVERSTOCK
5,79,22,OVERSTOCK
6,29,9,IN STOCK
7,58,1,OVERSTOCK
8,98,10,OVERSTOCK
9,4,2,LOW STOCK
10,91,2,OVERSTOCK
